In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

def calc_meissner(data, t, dt):

	I = np.zeros((4, len(t)))
	
	for i in range(len(t)):
		if(i!=0):
			dF_dt = np.abs(data[i] - data[i-1])/(t[i]-t[i-1])

			I[0, i] = I[0, i-1] + 1*dF_dt + (-I[0, i-1]*dt/(8*1* 1e-3)) 
			I[1, i] = I[1, i-1] + 0.24*dF_dt + (-(I[1, i-1] - 0.24*0.13)*dt/(200* 1e-3) )
			I[2, i] = I[2, i-1] + 0.07*dF_dt + (-I[2, i-1]*dt/(1744.6* 1e-3))
			I[3, i] = I[0, i] 

	return I[3,:]


def calc_merkel(data, t, dt):

	I = np.zeros((4, len(t)))
	
	for i in range(len(t)):
		if(i!=0):
			dF_dt = np.abs(data[i] - data[i-1])/(t[i]-t[i-1])

			if(dF_dt < 0):
				dF_dt = 0
			I[0, i] = I[0, i-1] + 0.74*dF_dt + (-I[0, i-1]*dt/(8*1* 1e-3)) 
			I[1, i] = I[1, i-1] + 0.24*dF_dt + (-(I[1, i-1] - 0.24*0.13)*dt/(200*1* 1e-3) )
			I[2, i] = I[2, i-1] + 0.07*dF_dt + (-I[2, i-1]*dt/(1744.6*1* 1e-3))
			I[3, i] = I[0, i] +  I[1, i] +  I[2, i] 

	return I[3,:]


path = "/Users/elast/OneDrive/デスクトップ/卒業研究/研究コード/"
dir_name = ["Al_board", "buta_omote", "buta_ura", "cork", "denim", "rubber_board", "washi", "wood_board"]

i = 1
j = 3

dt = 0.1*1e-3
dt_data = 0.1 * 1e-3


# plt.figure(figsize=(10, 5))
# plt.plot(t_array, in_data_0[0], label="0")
# plt.plot(t_array, in_data_0[1], label="1")
# plt.plot(t_array, in_data_0[2], label="2")
# plt.legend()
# plt.show()

clr =["r","g","b"]
fig,axes = plt.subplots(2,4,figsize=(20,10),squeeze=False)
plt.figure(figsize=(10, 5))
for i in range(8):
	for j in range(20):
		df = pd.read_table(glob.glob(path+"tactile_data/" +dir_name[i]+ f"/data_{int(j+2)}_*")[0], header=None)
		df_np = df.to_numpy().T
		in_data_0 =  df_np[:3, 3000:8000]
		nt = in_data_0.shape[1]
		T = nt*dt_data
		t_array = np.arange(nt) * dt_data
		for k in range(3):	
			axes[i//4,i%4].plot(t_array, in_data_0[k,:], color=clr[k], label=f"sensor {k+1}")
	axes[i//4,i%4].set_title(dir_name[i])
	axes[i//4,i%4].set_ylim([-0.4,0.4])
plt.show()



: 

In [ ]:
fig,axes = plt.subplots(2,4,figsize=(20,10),squeeze=False)
for i in range(8):
	for j in range(10):
		df = pd.read_table(glob.glob(path+"tactile_data/" +dir_name[i]+ f"/data_{int(j+1)}_*")[0], header=None)
		df_np = df.to_numpy().T
		in_data_0 =  df_np[:3, 3000:8000]
		nt = in_data_0.shape[1]
		T = nt*dt_data
		t_array = np.arange(nt) * dt_data
		input_current = np.zeros((6, nt))
		for k in range(3):
			in_data = in_data_0[k,:]
			I_merkel = calc_merkel(in_data, t_array, dt)
			I_meissner = calc_meissner(in_data, t_array, dt)
			input_current[k*2, :] = 0.01*I_merkel
			input_current[k*2+1, :] = 0.04*I_meissner

		#axes[i//4,i%4].plot(t_array, input_current[0])
		axes[i//4,i%4].plot(t_array, input_current[1])
		#axes[i//4,i%4].set_ylim([0,40])
	axes[i//4,i%4].set_title(dir_name[i])
	axes[i//4,i%4].set_ylim([0,40])


plt.show()

In [ ]:
from scipy import fftpack
from scipy import signal

n_sozai = 8
n_sample = 100
sample_for_cls = 80
sample_for_tst = 20
k = 0 #sensor number

sample_seq = np.load('sample_seq.npy')
max = len(sample_seq)
test_seq = sample_seq[100:max]

dim = 10*44

feature_vec = np.zeros([n_sozai*n_sample,dim])
X = np.zeros([n_sozai*n_sample,dim])
Y = np.zeros([n_sozai*sample_for_cls,dim])
Y_std = np.zeros([n_sozai*sample_for_cls,dim])
Z = np.zeros([n_sozai*sample_for_tst,dim])
Z_std = np.zeros([n_sozai*sample_for_tst,dim])
vec_ave = np.zeros([n_sozai,dim])
vec_cov = np.zeros([n_sozai,dim,dim])
cov_mtrx = np.zeros([n_sozai,dim,dim])


for i in range(n_sozai):
    for j in range(n_sample):
        df = pd.read_table(glob.glob(path+"tactile_data/" +dir_name[i]+ f"/data_{int(test_seq[j])}_*")[0], header=None)
        df_np = df.to_numpy().T
        data =  df_np[:3, 3000:8000]
        fs = 10000
        f,t,Sxx = signal.spectrogram(data[k], fs, nperseg=128)
        X[i*n_sample+j,:] = 10*np.log10(Sxx[0:10,:].reshape(1,dim))
        if j < sample_for_cls:
            Y[i*sample_for_cls+j,:] = 10*np.log10(Sxx[0:10,:].reshape(1,dim))
        else:
            Z[i*sample_for_tst+j-sample_for_cls,:] = 10*np.log10(Sxx[0:10,:].reshape(1,dim))
        
        #feature_vec[i*n_sample+j,:] = 10*np.log10(Sxx[0:10,:].reshape(1,dim))
    vec_ave[i,:] = np.mean(Y[i*sample_for_cls:(i+1)*sample_for_cls,:], axis=0)
    vec_cov[i, :, :] = np.cov(Y[i*sample_for_cls:(i+1)*sample_for_cls, :].T)
    #vec_ave[i,:] = np.mean(feature_vec[i*n_sample:i*n_sample+sample_for_cls,:], axis=0)
    #vec_cov[i, :, :] = np.cov(feature_vec[i*n_sample:i*n_sample+sample_for_cls, :].T)

for i in range(n_sozai):
    cov_mtrx[i, :, :] = np.linalg.pinv(vec_cov[i, :, :])

seigo = np.zeros([n_sozai*sample_for_tst, 2])

for i in range(n_sozai):
    for j in range(sample_for_tst):
        diff_sozai = np.zeros(n_sozai)
        
        for m in range(n_sozai):              
            data_minus_ave = Z[i*sample_for_tst+j,:] - vec_ave[m,:]
            diff_sozai[m] = np.sqrt( (data_minus_ave.reshape([1,-1]) @ cov_mtrx[m, :, :]) @ data_minus_ave.reshape([-1,1]))

        seigo[i*sample_for_tst + j, 0] = int(i)
        seigo[i*sample_for_tst + j, 1] = int(np.argmin(diff_sozai))

conf_mtrx = np.zeros([8,8])
y_true = seigo[:,0]
y_pred = seigo[:,1]
count = 0
for i in range(n_sozai*sample_for_tst):
    conf_mtrx[i//sample_for_tst,int(y_pred[i])] += 1
    if y_true[i] == y_pred[i]:
        count += 1
accuracy8 = count / (n_sozai*sample_for_tst)
print(accuracy8)
print(conf_mtrx)


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_std = scaler.fit_transform(X.T).T
for i in range(n_sozai):
    for j in range(n_sample):
        if j < sample_for_cls:
            Y_std[i*sample_for_cls+j,:] = X_std[i*n_sample+j,:]
        else:
            Z_std[i*sample_for_tst+j-sample_for_cls,:] = X_std[i*n_sample+j,:]


cov_mtrx=np.cov(X_std.T)
eig_values, eig_vectors = np.linalg.eig(cov_mtrx)
Y_pca = np.dot(Y_std, eig_vectors[:,:3])
Z_pca = np.dot(Z_std, eig_vectors[:,:3])


fig,axes = plt.subplots(2,4,figsize=(20,10),squeeze=False,subplot_kw=dict(projection='3d'))
for i in range(8):
    start = i*sample_for_cls
    end = start + 30
    axes[i//4,i%4].scatter(Y_pca[start:end, 0], Y_pca[start:end, 1], Y_pca[start:end, 2])
    axes[i//4,i%4].scatter(Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 0], Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 1], Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 2], color="r")

    axes[i//4,i%4].set_title(dir_name[i])
    axes[i//4,i%4].set_xlim([-17,0])
    axes[i//4,i%4].set_ylim([-10,10])
    axes[i//4,i%4].set_zlim([-8,10])
    #print(start,end)
plt.show()


print(10*np.log10(Sxx[1,:]))
plt.figure()
plt.pcolormesh(t,f,10*np.log10(Sxx))
plt.colorbar(label='dB')
plt.ylim([0,200])
plt.show()

In [ ]:
n_sozai = 8
n_sample = 100
sample_for_cls = 80
sample_for_tst = 20
k = 0 #sensor number

dt_data = 0.1 * 1e-3

sample_seq = np.load('sample_seq.npy')
max = len(sample_seq)
test_seq = sample_seq[100:max]

dim = 5*44

feature_vec = np.zeros([n_sozai*n_sample,dim])
X = np.zeros([n_sozai*n_sample,dim])
Y = np.zeros([n_sozai*sample_for_cls,dim])
Y_std = np.zeros([n_sozai*sample_for_cls,dim])
Z = np.zeros([n_sozai*sample_for_tst,dim])
Z_std = np.zeros([n_sozai*sample_for_tst,dim])
vec_ave = np.zeros([n_sozai,dim])
vec_cov = np.zeros([n_sozai,dim,dim])
cov_mtrx = np.zeros([n_sozai,dim,dim])


for i in range(n_sozai):
    for j in range(n_sample):
        df = pd.read_table(glob.glob(path+"tactile_data/" +dir_name[i]+ f"/data_{int(test_seq[j])}_*")[0], header=None)
        df_np = df.to_numpy().T
        data =  df_np[:3, 3000:8000]

        nt = data.shape[1]
        T = nt*dt_data
        t_array = np.arange(nt) * dt_data
        input_current = np.zeros((2, nt))
		# for k in range(3):
		# 	in_data = data[k,:]
		# 	I_merkel = calc_merkel(in_data, t_array, dt)
		# 	I_meissner = calc_meissner(in_data, t_array, dt)
		# 	input_current[k*2, :] = 0.01*I_merkel
		# 	input_current[k*2+1, :] = 0.04*I_meissner

        in_data = data[0,:]
        I_merkel = calc_merkel(in_data, t_array, dt)
        I_meissner = calc_meissner(in_data, t_array, dt)
        input_current[0, :] = 0.01*I_merkel
        input_current[1, :] = 0.04*I_meissner
               
        fs = 10000
        #f,t,Sxx = signal.spectrogram(input_current[0], fs, nperseg=128)
        #feature_vec[i*n_sample+j,:dim] = 10*np.log10(Sxx[0:10,:].reshape(1,dim))
        f,t,Sxx = signal.spectrogram(input_current[0], fs, nperseg=128)
        X[i*n_sample+j,:] = 10*np.log10(Sxx[0:5,:].reshape(1,dim))
        if j < sample_for_cls:
            Y[i*sample_for_cls+j,:] = 10*np.log10(Sxx[0:5,:].reshape(1,dim))
        else:
            Z[i*sample_for_tst+j-sample_for_cls,:] = 10*np.log10(Sxx[0:5,:].reshape(1,dim))

    vec_ave[i,:] = np.mean(Y[i*sample_for_cls:(i+1)*sample_for_cls,:], axis=0)
    vec_cov[i, :, :] = np.cov(Y[i*sample_for_cls:(i+1)*sample_for_cls, :].T)

for i in range(n_sozai):
    cov_mtrx[i, :, :] = np.linalg.pinv(vec_cov[i, :, :])

seigo = np.zeros([n_sozai*sample_for_tst, 2])
for i in range(n_sozai):
    for j in range(sample_for_tst):
        diff_sozai = np.zeros(n_sozai)
        
        for m in range(n_sozai):              
            data_minus_ave = Z[i*sample_for_tst+j,:] - vec_ave[m,:]
            diff_sozai[m] = np.sqrt( (data_minus_ave.reshape([1,-1]) @ cov_mtrx[m, :, :]) @ data_minus_ave.reshape([-1,1]))

        seigo[i*sample_for_tst + j, 0] = int(i)
        seigo[i*sample_for_tst + j, 1] = int(np.argmin(diff_sozai))

conf_mtrx = np.zeros([8,8])
y_true = seigo[:,0]
y_pred = seigo[:,1]
count = 0
for i in range(n_sozai*sample_for_tst):
    conf_mtrx[i//sample_for_tst,int(y_pred[i])] += 1
    if y_true[i] == y_pred[i]:
        count += 1
accuracy8 = count / (n_sozai*sample_for_tst)
print(accuracy8)
print(conf_mtrx)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_std = scaler.fit_transform(X.T).T
for i in range(n_sozai):
    for j in range(n_sample):
        if j < sample_for_cls:
            Y_std[i*sample_for_cls+j,:] = X_std[i*n_sample+j,:]
        else:
            Z_std[i*sample_for_tst+j-sample_for_cls,:] = X_std[i*n_sample+j,:]


cov_mtrx=np.cov(X_std.T)
eig_values, eig_vectors = np.linalg.eig(cov_mtrx)
Y_pca = np.dot(Y_std, eig_vectors[:,:3])
Z_pca = np.dot(Z_std, eig_vectors[:,:3])


fig,axes = plt.subplots(2,4,figsize=(20,10),squeeze=False,subplot_kw=dict(projection='3d'))
for i in range(8):
    start = i*sample_for_cls
    end = start + 30
    axes[i//4,i%4].scatter(Y_pca[start:end, 0], Y_pca[start:end, 1], Y_pca[start:end, 2])
    axes[i//4,i%4].scatter(Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 0], Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 1], Z_pca[i*sample_for_tst:(i+1)*sample_for_tst, 2], color="r")

    axes[i//4,i%4].set_title(dir_name[i])
    axes[i//4,i%4].set_xlim([-6,8])
    axes[i//4,i%4].set_ylim([-8,2])
    axes[i//4,i%4].set_zlim([-6,2])
    #print(start,end)
plt.show()